In [ ]:
from sodapy import Socrata
import pandas as pd
from itertools import islice
import os
from pathlib import Path
import requests

In [4]:
# If the CTA data exists, read it in
if Path("output/cta_ridership.parquet").is_file():
    cta_df = pd.read_parquet("output/cta_ridership.parquet")
else:
    # If the CTA data does not exist, download it from Socrata
    client = Socrata(
        "data.cityofchicago.org",
        timeout=1000,
        app_token=None
    )

    cta_data = client.get_all('5neh-572f')

    chunk_size = 10_000
    chunks = []

    while True:
        chunk = list(islice(cta_data, chunk_size))
        print('Grabbing chunk of data...')
        if not chunk:
            break
        chunks.append(pd.DataFrame(chunk))

    cta_df = pd.concat(chunks, ignore_index=True)

    # And save as a flat file
    os.makedirs("output", exist_ok=True)
    cta_df.to_parquet(path='output/cta_ridership.parquet', engine='fastparquet', index=False)

In [ ]:
cta_df.tail(10)

,station_id,stationname,date,daytype,rides
1285283,41450,Chicago/State,2025-08-31,U,8115
1285284,41460,Irving Park-Brown,2025-08-31,U,1082
1285285,41480,Western-Brown,2025-08-31,U,1340
1285286,41490,Harrison,2025-08-31,U,2756
1285287,41500,Montrose-Brown,2025-08-31,U,1022
1285288,41510,Morgan-Lake,2025-08-31,U,3702
1285289,41660,Lake/State,2025-08-31,U,10951
1285290,41670,Conservatory,2025-08-31,U,558
1285291,41680,Oakton-Skokie,2025-08-31,U,250
1285292,41690,Cermak-McCormick Place,2025-08-31,U,1459


In [ ]:
# Check that the file is up-to-date
# If the data exists and the last row number is smaller than the last row number on Socrata, re-download
nrow_in_data = cta_df.shape[0]
print(f'Number of rows in the data: {nrow_in_data}')

# Check against Socrata
url = "https://data.cityofchicago.org/resource/5neh-572f.json"
params = {
    "$select": "count(*)"
}

data_socrata_json = requests.get(url, params=params).json()
nrow_in_socrata = int(data_socrata_json[0]['count'])

if nrow_in_data == nrow_in_socrata:
    print('The local data is up-to-date.')
else:
    print('Downloading new data...')
    params_download = {
        "$offset": nrow_in_data
    }

    data_new = requests.get(url, params=params_download).json()
    df_new = pd.DataFrame(data_new)
    cta_df = pd.concat([cta_df, df_new], ignore_index=True)

    # Save the flat file
    os.makedirs("output", exist_ok=True)
    cta_df.to_parquet(path='output/cta_ridership.parquet', engine='fastparquet', index=False)


Number of rows in the data: 1285295
The local data is up-to-date.
